---
## Section 3.2 — Zero-Shot MATH Baseline

### Problem `math_baseline`

**(a)** Write a script to evaluate Qwen 2.5 Math 1.5B zero-shot on MATH. It should (1) load the MATH validation examples, (2) format them with the `r1_zero` prompt, (3) generate outputs with a vLLM model, (4) calculate evaluation metrics, and (5) serialize examples, generations, and scores to disk.

**(b)** Run the script. How many generations fall into each category: (1) format=1 & answer=1, (2) format=1 & answer=0, (3) format=0 & answer=0? Inspect ≥10 cases per failing category — is the issue the model or the parser?

**(c)** How well does Qwen 2.5 Math 1.5B perform zero-shot on MATH?

#### evaluation script

[`ece405_assignment3/cs336_alignment/math_baseline.py`](../cs336_alignment/math_baseline.py).

`evaluate_vllm(vllm_model, reward_fn, prompts, eval_sampling_params, *, ground_truths, output_path, extras)`
1. Calls `vllm_model.generate(prompts, eval_sampling_params)` for examples
2. Scores each response with `reward_fn(response, ground_truth)` → `{format_reward, answer_reward, reward}`.
3. Categorises each result as `format_correct_answer_correct`, `format_correct_answer_wrong`, or `format_wrong`.
4. Writes a per-example `generations.jsonl` and a `metrics.json` summary to `--output-dir`.

#### (b) Category analysis

In [6]:
import json, random
from collections import Counter, defaultdict
from pathlib import Path

def _find_results_dir(name: str) -> Path:
    candidates = [
        Path(f"results/{name}"),
        Path(f"../results/{name}"),
        Path(f"ece405_assignment3/results/{name}"),
        Path.home() / f"projects/LLM-from-scratch/ece405_assignment3/results/{name}",
    ]
    for c in candidates:
        if c.is_dir():
            return c.resolve()
    raise FileNotFoundError(f"Cannot locate results/{name}. Tried:\n" + "\n".join(str(c.resolve()) for c in candidates))

RESULTS_DIR = _find_results_dir("math_baseline")
print(f"Loading from {RESULTS_DIR}")

with open(RESULTS_DIR / "generations.jsonl") as f:
    results = [json.loads(line) for line in f if line.strip()]

by_cat = defaultdict(list)
for r in results:
    by_cat[r["category"]].append(r)

counts = Counter(r["category"] for r in results)
print(f"Total examples: {len(results)}\n")
print("Category breakdown:")
for cat, n in counts.most_common():
    print(f"  {cat:45s}  {n:5d}  ({n/len(results):.1%})")

def show_examples(category, n=10, seed=0):
    sample = random.Random(seed).sample(by_cat[category], min(n, len(by_cat[category])))
    for i, ex in enumerate(sample, 1):
        print(f"\n--- {i}. ---")
        print("RESPONSE:", ex["response"][:500])
        print(f"format_reward={ex['format_reward']}  answer_reward={ex['answer_reward']}")
        print("-" * 70)

print("\n\n=== format_wrong (format_reward=0) — 10 examples ===")
show_examples("format_wrong", n=10)

print("\n\n=== format_correct_answer_wrong (format=1, answer=0) — 10 examples ===")
show_examples("format_correct_answer_wrong", n=10)

FileNotFoundError: Cannot locate results/math_baseline. Tried:
/content/results/math_baseline
/results/math_baseline
/content/ece405_assignment3/results/math_baseline
/root/projects/LLM-from-scratch/ece405_assignment3/results/math_baseline

**Commentary — format_reward = 0 cases (85.8% of examples):**

The base model almost never produces the required `<answer>...</answer>` structure. Common failure modes observed in sampled outputs:

- **Truncation before the answer tag**: Many responses hit the `max_tokens=1024` limit mid-reasoning and never emit `<answer>` at all.
- **LaTeX `\boxed{}` instead of XML tags**: The base model was pretrained to output `\boxed{answer}` and has no signal to switch to the `<answer>` schema.
- **Malformed XML**: Occasionally the model produces `</think>` but wraps its answer in `<answer>$val$<br></answer>` (with embedded HTML), or uses garbled tags that fail the format check.
- **Continues past the stop token**: Some responses emit a valid-looking `<answer>` block embedded in longer output without ever triggering the `</answer>` stop string.

The root cause is that Qwen 2.5 Math 1.5B (base, not instruct) was never trained on the `<think>…</think><answer>…</answer>` format — zero-shot compliance is very low. **The issue is the model, not the parser.**

**Commentary — format_reward = 1, answer_reward = 0 cases (11.6% of examples):**

When the model does emit a properly-formed `<answer>…</answer>` block, it still gets the answer wrong ~82% of the time (581/710 format-correct examples). Observed patterns:

- **Arithmetic mistakes**: Correct strategy, wrong computation (e.g., correct formula, mis-evaluated intermediate step).
- **Wrong strategy**: Applies a valid method to the wrong subproblem or misreads the question.
- **Garbled content inside valid tags**: Occasionally outputs a near-English description (e.g., `"4*4=16 different positive two-digit integers…"`) inside `<answer>` tags, which doesn't match the expected simplified form.

The 11.6% format-correct-but-wrong rate shows the model understands *some* problems well enough to reach a numeric answer — it just computes it incorrectly. This sets a practical ceiling: format-only training (SFT/GRPO on format) can raise compliance substantially, but high accuracy also requires improved reasoning.

#### (c) Deliverable — zero-shot baseline performance

In [ ]:
import json
from pathlib import Path

def _find_results_dir(name: str) -> Path:
    candidates = [
        Path(f"results/{name}"),
        Path(f"../results/{name}"),
        Path(f"ece405_assignment3/results/{name}"),
        Path.home() / f"projects/LLM-from-scratch/ece405_assignment3/results/{name}",
    ]
    for c in candidates:
        if c.is_dir():
            return c.resolve()
    raise FileNotFoundError(f"Cannot locate results/{name}. Tried:\n" + "\n".join(str(c.resolve()) for c in candidates))

RESULTS_DIR = _find_results_dir("math_baseline")
metrics = json.loads((RESULTS_DIR / "metrics.json").read_text())
print(json.dumps(metrics, indent=2))
print(f"\nZero-shot accuracy : {metrics['accuracy']:.1%}")
print(f"Format compliance  : {metrics['format_rate']:.1%}")

Qwen 2.5 Math 1.5B achieves **2.6% accuracy** (129/5000) with only **14.2% format compliance** zero-shot on the MATH validation set. The dominant failure mode (85.8% of examples) is that the base model never produces the required `<answer>…</answer>` XML structure, since it was pretrained to emit `\boxed{}` answers rather than the r1-zero prompt schema.

---
## Section 4 — Supervised Finetuning for MATH

SFT Qwen2.5-Math-1.5B Base on R1 reasoning traces. All adapters are wired in `tests/adapters.py`. Tests: `tests/test_sft.py`.

### Problem `tokenize_prompt_and_output` — 2 points

Implement `tokenize_prompt_and_output(prompt_strs, output_strs, tokenizer)`. Returns `input_ids`, `labels`, `response_mask` (1 on response tokens, 0 on prompt/padding).

**Deliverable:** implementation in `cs336_alignment/sft.py`. Test: `uv run pytest -k test_tokenize_prompt_and_output`

Implemented in [`cs336_alignment/sft.py`](../cs336_alignment/sft.py) — `tokenize_prompt_and_output`.

### Problem `compute_entropy` — 1 point

`compute_entropy(logits)` → per-token entropy over vocab dim, using logsumexp for numerical stability.

**Deliverable:** implementation. Test: `uv run pytest -k test_compute_entropy`

Implemented in [`cs336_alignment/sft.py`](../cs336_alignment/sft.py) — `compute_entropy`.

### Problem `get_response_log_probs` — 2 points

`get_response_log_probs(model, input_ids, labels, return_token_entropy=False)` → `{"log_probs": ..., "token_entropy": ...}`.

**Deliverable:** implementation. Test: `uv run pytest -k test_get_response_log_probs`

Implemented in [`cs336_alignment/sft.py`](../cs336_alignment/sft.py) — `get_response_log_probs`.

### Problem `masked_normalize` — 1 point

`masked_normalize(tensor, mask, normalize_constant, dim=None)` — sum masked elements along dim, divide by normalize_constant.

**Deliverable:** implementation. Test: `uv run pytest -k test_masked_normalize`

Implemented in [`cs336_alignment/sft.py`](../cs336_alignment/sft.py) — `masked_normalize`.

### Problem `sft_microbatch_train_step` — 3 points

`sft_microbatch_train_step(policy_log_probs, response_mask, gradient_accumulation_steps, normalize_constant=1.0)` — NLL loss on response tokens, divides by `gradient_accumulation_steps`, calls `loss.backward()`, returns `(loss, metadata)`.

**Deliverable:** implementation. Test: `uv run pytest -k test_sft_microbatch_train_step`

Implemented in [`cs336_alignment/sft.py`](../cs336_alignment/sft.py) — `sft_microbatch_train_step`.

### Problem `log_generations` — 1 point

`log_generations(...)` — log prompt, response, ground truth, reward (format/answer/total), average token entropy, and avg response length (overall, correct, incorrect).

**Deliverable:** implementation.

Implemented in [`cs336_alignment/sft.py`](../cs336_alignment/sft.py) — `log_generations`.

### Problem `sft_experiment` — 2 points

1. Run SFT on `sft.jsonl` with dataset sizes `{128, 256, 512, 1024, full}`. Tune lr/batch to ≥15% validation accuracy on full data. **Deliverable:** validation accuracy curves vs dataset size.
2. Filter SFT examples to only correct-answer ones; rerun SFT on the filtered full set. **Deliverable:** filtered dataset size + validation accuracy curve.

ECE405 deviation #6: ~30 min training budget.

In [ ]:
# TODO: run SFT experiment and display results

---
## Section 5 — Expert Iteration for MATH

EI: sample G rollouts per question, keep correct ones, SFT on them, repeat for `n_ei_steps`.

### Problem `expert_iteration_experiment` — 2 points

Run EI on MATH with `n_ei_steps=5`. Vary G ∈ rollouts and epochs in SFT step (≥2 configs each). Batch size `D_b` ∈ `{512, 1024, 2048}`.

**Deliverables:** validation accuracy curves per config; model ≥15% accuracy; 2-sentence comparison vs SFT; entropy plot over training.

In [ ]:
# TODO: run expert iteration and display results

---
## Section 6 — Primer on Policy Gradients

*(Reading section — no problems.)*

Key ideas: LM as categorical policy; REINFORCE gradient; variance-reducing baseline; off-policy importance weights.

---
## Section 7 — Group Relative Policy Optimization

GRPO: sample G outputs per question, group-normalize advantages, off-policy update with PPO-style clipping. Tests: `tests/test_grpo.py`.

### Problem `compute_group_normalized_rewards` — 2 points

`compute_group_normalized_rewards(reward_fn, rollout_responses, repeated_ground_truths, group_size, advantage_eps, normalize_by_std)` → `(advantages, raw_rewards, metadata)`. If `normalize_by_std=False`, advantage is $r - \bar{r}$ (Dr. GRPO).

**Deliverable:** implementation. Test: `uv run pytest -k test_compute_group_normalized_rewards`

Implemented in [`cs336_alignment/grpo.py`](../cs336_alignment/grpo.py) — `compute_group_normalized_rewards`.

### Problem `compute_naive_policy_gradient_loss` — 1 point

`compute_naive_policy_gradient_loss(raw_rewards_or_advantages, policy_log_probs)` → per-token loss $-A_t \cdot \log \pi(o_t|\ldots)$.

**Deliverable:** implementation. Test: `uv run pytest -k test_compute_naive_policy_gradient_loss`

Implemented in [`cs336_alignment/grpo.py`](../cs336_alignment/grpo.py) — `compute_naive_policy_gradient_loss`.

### Problem `compute_grpo_clip_loss` — 2 points

`compute_grpo_clip_loss(advantages, policy_log_probs, old_log_probs, cliprange)` → clipped PG loss + clip-fraction metadata.

**Deliverable:** implementation. Test: `uv run pytest -k test_compute_grpo_clip_loss`

Implemented in [`cs336_alignment/grpo.py`](../cs336_alignment/grpo.py) — `compute_grpo_clip_loss`.

### Problem `compute_policy_gradient_loss` — 1 point

Wrapper dispatching on `loss_type ∈ {no_baseline, reinforce_with_baseline, grpo_clip}`.

**Deliverable:** implementation. Test: `uv run pytest -k test_compute_policy_gradient_loss`

Implemented in [`cs336_alignment/grpo.py`](../cs336_alignment/grpo.py) — `compute_policy_gradient_loss`.

### Problem `masked_mean` — 1 point

`masked_mean(tensor, mask, dim=None)` → mean over masked positions.

**Deliverable:** implementation. Test: `uv run pytest -k test_masked_mean`

Implemented in [`cs336_alignment/grpo.py`](../cs336_alignment/grpo.py) — `masked_mean`.

### Problem `grpo_microbatch_train_step` — 3 points

`grpo_microbatch_train_step(policy_log_probs, response_mask, gradient_accumulation_steps, loss_type, ...)` — per-token loss → masked_mean over response → mean over batch → divide by `gradient_accumulation_steps` → backward.

**Deliverable:** implementation. Test: `uv run pytest -k test_grpo_microbatch_train_step`

Implemented in [`cs336_alignment/grpo.py`](../cs336_alignment/grpo.py) — `grpo_microbatch_train_step`.

### Problem `grpo_train_loop` — 5 points

Full GRPO training loop. Default hyperparameters: `n_grpo_steps=200`, `lr=1e-5`, `rollout_batch_size=256`, `group_size=8`, `gradient_accumulation_steps=128`, `loss_type=reinforce_with_baseline`.

**Deliverable:** training script + validation reward curve + sample rollouts over time.

Implemented in [`cs336_alignment/train_grpo.py`](../cs336_alignment/train_grpo.py). Logs to wandb (`ece405-grpo` project).

Run on Koa:
```bash
koa submit scripts/run_grpo.sh
```

In [ ]:
# TODO: display validation reward curve from wandb / results dir

---
## Section 8 — GRPO Experiments

### Problem `grpo_learning_rate` — 2 points

Sweep learning rate. **Deliverable:** validation reward curves; model achieving ≥25% MATH accuracy on at least one LR.

In [ ]:
# TODO: LR sweep results

### Problem `grpo_baselines` — 2 points

Compare `no_baseline` vs `reinforce_with_baseline`. **Deliverable:** reward curves and commentary.

In [ ]:
# TODO: baseline ablation results

### Problem `think_about_length_normalization` — 1 point

Compare `masked_mean` vs `masked_normalize` (constant = max generation length). Pros/cons of each?

**Deliverable:** written analysis.

> *(Written response — TODO)*

### Problem `grpo_length_normalization` — 2 points

Empirical: GRPO with `masked_mean` vs `masked_normalize`. Report curves and gradient norm stability.

In [ ]:
# TODO: length normalization ablation results

### Problem `grpo_group_standard_deviation` — 2 points

Compare `use_std_normalization=True` vs `False`. Report curves and stability commentary.

In [ ]:
# TODO: std normalization ablation results

### Problem `grpo_off_policy` — implementation

Off-policy GRPO with multiple epochs per rollout batch, `old_log_probs` computed once, `loss_type="grpo_clip"`.

**Deliverable:** implementation in `cs336_alignment/train_grpo.py` (already wired via `--loss-type grpo_clip --epochs-per-rollout-batch N`).

### Problem `grpo_off_policy_sweep` — 4 points

Fix `rollout_batch_size=256`. Sweep `epochs_per_rollout_batch` × `train_batch_size`. Compare on-policy baseline by validation accuracy and wall-clock.

In [ ]:
# TODO: off-policy sweep results

### Problem `grpo_off_policy_clip_ablation` — 2 points

Add `loss_type="grpo_no_clip"` ($-\pi/\pi_{old} \cdot A_t$). Compare to clipped: entropy, response length, gradient norm.

In [ ]:
# TODO: no-clip ablation results

### Problem `grpo_prompt_ablation` — 2 points

Train with `question_only.prompt` + `question_only_reward_fn`. Compare to R1-Zero prompt: entropy, response length, gradient norm.

In [ ]:
# TODO: prompt ablation results

---
## Section 9 — Leaderboard

### Problem `leaderboard` — 16 points

Maximize MATH validation accuracy in ≤4 hours on 2 H100s. No extra data or models. Use R1-Zero prompt + `r1_zero_reward_fn`; temperature 1.0, max_tokens 1024 for validation.

**Deliverable:** validation accuracy + screenshot of accuracy vs wall-clock.

In [ ]:
# TODO: leaderboard run results

---
## Optional Supplement — Instruction Tuning, MMLU/GSM8K, DPO, Safety

Covers 2024-style problems (ECE405 deviations #4, #6, #7). See `cs336_spring2024_assignment5_alignment.pdf` and `cs336_spring2025_assignment5_supplement_safety_rlhf.pdf`.

### Problem `sft` — Instruction Tuning

Pack Alpaca-style examples into fixed-length sequences; train Qwen2.5-0.5B for ~30 min.

**Deliverable:** `get_packed_sft_dataset`, `run_iterate_batches` in `tests/adapters.py`.

In [ ]:
# TODO

### Problem `mmlu_baseline`

Zero-shot MMLU. Implement `run_parse_mmlu_response` to extract A/B/C/D.

In [ ]:
# TODO

### Problem `gsm8k_baseline`

Zero-shot GSM8K. `run_parse_gsm8k_response` returns the last numeric token.

In [ ]:
# TODO

### Problem `alpaca_eval_baseline`

ECE405 deviation #4: edit `scripts/alpaca_eval_vllm_llama3_70b_fn` so `model_name` points at the local Qwen2.5-3B-Instruct dir.

In [ ]:
# TODO

### Problem `sst_baseline`

Run `scripts/evaluate_safety.py` with local Qwen2.5-3B-Instruct against SimpleSafetyTests.

In [ ]:
# TODO

### Problem `dpo_training`

DPO on Anthropic HH for ~30 min. Single GPU: query reference and trained model consecutively.

**Deliverable:** `run_compute_per_instance_dpo_loss` in `tests/adapters.py`.

In [ ]:
# TODO

---
## Submission

Per ECE405 deviation #2:
- Submit the report via the [Google Form](https://docs.google.com/forms/d/e/1FAIpQLScJg_QkwjKux3xKeM-EOmZyvA6zlbVIrf_lxN_qoCFoxdqTrg/viewform).
- Include a link to your GitHub branch plus any wandb run links.
- Code does **not** need to be attached.
- **No leaderboard submission required.**